# 🕰️ Notebook 1: Lamport Clocks — and why they're not enough

**The big question:** *"In a distributed system, what does it even mean to say event A happened before event B?"*

In a single program, the answer is obvious: whichever statement ran first. But across many machines, nothing is obvious. Each machine has its own clock and its own idea of "now."

In this notebook we'll:
1. See why **wall-clock time** (`time.time()`) is a terrible way to order distributed events.
2. Implement **Lamport clocks** — Leslie Lamport's 1978 trick for ordering events using only a counter.
3. Discover a limitation: Lamport clocks can tell you *an* order, but they can't tell you whether two events are actually **concurrent** (unrelated). That's what Notebook 2 fixes with vector clocks.

## Learning objectives
- Understand the **happens-before** relation (denoted `→`).
- Implement a Lamport clock from scratch.
- See a scenario where Lamport clocks are misleading about concurrency.


## 🔧 Setup

This lab has its own virtual environment. To run the cells:

1. Run `uv sync` in `02-distributed-primitives/vector-clocks/` (only needed once).
2. In VS Code, click the kernel picker (top-right of this notebook) and pick the `.venv` Python.
3. If `.venv` doesn't appear, reload VS Code: `Cmd+Shift+P` → **Reload Window**.

No Docker, no network services — everything is pure Python.

## 1. 😱 The bad baseline: wall-clock time

Imagine three servers — `A`, `B`, `C` — each with its own clock. Clocks drift. Even NTP-synced machines can be off by tens of milliseconds. Some cloud VMs have been observed **seconds** apart.

Below we simulate this **deterministically**: no real `sleep`, no flakiness. Each process has a fixed clock skew, and messages take some network time to arrive. Then we tag each event with that process's wall-clock at the moment it happened.

We'll do a simple causal chain:
- `A` writes `x = 1` and sends it to `B`.
- `B` receives, then writes `y = 2` and sends to `C`.
- `C` receives, then writes `z = 3`.

Everyone knows, by construction, that `A` → `B` → `C`. Let's see if wall-clock timestamps agree.

In [ ]:
# A tiny simulator. "now" is fake time — no real sleeping.
class WallClockProcess:
    def __init__(self, name, skew):
        self.name = name
        self.skew = skew           # how far off from "true" time this machine is
        self.true_time = 0.0       # shared simulated true time
        self.log = []              # list of (wall_time_reported, event)

    def now(self):
        # Each process only sees its own skewed clock.
        return self.true_time + self.skew

    def tick(self, dt):
        # Advance true time by dt seconds (simulating work / network).
        self.true_time += dt

    def event(self, label):
        self.log.append((round(self.now(), 3), f"{self.name}: {label}"))


# Three processes with different clock skews (in seconds).
A = WallClockProcess("A", skew=+0.00)   # on time
B = WallClockProcess("B", skew=-0.50)   # 500 ms behind
C = WallClockProcess("C", skew=+0.20)   # 200 ms ahead

# Helper: advance the shared "true" time for every process at once.
def advance(dt):
    for p in (A, B, C):
        p.tick(dt)

# --- Run the causal chain ---
A.event("write x=1")          # t=0.00s true
advance(0.05)                  # 50ms goes by while message travels
B.event("recv x=1")
B.event("write y=2")
advance(0.05)
C.event("recv y=2")
C.event("write z=3")

# Merge everyone's log and sort by their reported wall-clock time.
merged = sorted(A.log + B.log + C.log, key=lambda x: x[0])
for t, ev in merged:
    print(f"wall={t:.3f}s  {ev}")

# By construction A's write CAUSED B's receive, which CAUSED C's receive.
# If wall-clock time were a valid ordering, that chain would appear in order.
order = [ev for _, ev in merged]
pos = {name: i for i, name in enumerate(order)}
assert pos["B: recv x=1"] < pos["A: write x=1"], "expected the causal chain to be inverted"
print("\n💥 B's receive is timestamped BEFORE the send that caused it.")
print("   No message travelled backwards; B's clock is simply 500 ms behind.")


### 🔎 What just happened?

Look at the sorted-by-wall-clock timeline. Because `B`'s clock is behind and `C`'s is ahead, **`B`'s receive of `x=1` may look like it happened before `A` even sent it**, and `C`'s events may look like they happened before `B`'s.

> **Bad practice:** using wall-clock time to order events across machines.

Wall-clock time can't be trusted because clocks drift. Worse: even if clocks were perfectly synchronized, they still wouldn't tell us *why* two events are related (did one cause the other, or did they just happen at similar moments?).

We need a clock built from **causality**, not from seconds.

## 2. 💡 Lamport's idea: count causes, not seconds

Leslie Lamport (1978) proposed the **happens-before** relation `→`:

- If event `A` happens before event `B` **in the same process**, then `A → B`.
- If `A` is the sending of a message and `B` is its receive, then `A → B`.
- If `A → B` and `B → C`, then `A → C` (transitive).

Events that are **not** related by `→` are **concurrent** (written `A ∥ B`).

A **Lamport clock** is a single counter per process that respects `→`:

1. Before any local event, increment your clock.
2. When sending a message, attach your current clock value.
3. When receiving a message with timestamp `t`, set `clock = max(local, t) + 1`.

**Guarantee:** if `A → B` then `L(A) < L(B)`.

⚠️ The converse is **not** true. Two events can have `L(A) < L(B)` and still be concurrent.

In [ ]:
class LamportProcess:
    def __init__(self, name):
        self.name = name
        self.clock = 0
        self.history = []   # (lamport_time, description)

    def local(self, label):
        self.clock += 1
        self.history.append((self.clock, f"{self.name}: {label}"))

    def send(self, target, label):
        self.clock += 1
        self.history.append((self.clock, f"{self.name}: send '{label}' -> {target.name}"))
        return self.clock, label

    def recv(self, sender_clock, label):
        self.clock = max(self.clock, sender_clock) + 1
        self.history.append((self.clock, f"{self.name}: recv '{label}'"))


A = LamportProcess("A")
B = LamportProcess("B")
C = LamportProcess("C")

# Same causal chain as before.
A.local("write x=1")
m1 = A.send(B, "x=1"); B.recv(*m1)
B.local("write y=2")
m2 = B.send(C, "y=2"); C.recv(*m2)
C.local("write z=3")

# ALSO: two events with no causal link between them.
A.local("indep A1")   # A does something, never tells C about it
C.local("indep C1")   # C does something, never tells A about it

# Print all events sorted by Lamport timestamp.
for t, ev in sorted(A.history + B.history + C.history):
    print(f"L={t:2d}  {ev}")

L = {ev: t for t, ev in A.history + B.history + C.history}

# What Lamport GUARANTEES: if A → B then L(A) < L(B). Check the whole causal chain.
chain = ["A: write x=1", "A: send 'x=1' -> B", "B: recv 'x=1'", "B: write y=2",
         "B: send 'y=2' -> C", "C: recv 'y=2'", "C: write z=3"]
assert all(L[a] < L[b] for a, b in zip(chain, chain[1:])), \
    "Lamport must respect happens-before"

# What it does NOT guarantee: the converse. A1 and C1 are genuinely concurrent —
# no message ever connected them — yet Lamport assigns them different numbers and
# so implies an order that does not exist.
assert L["A: indep A1"] != L["C: indep C1"]
print(f"\n✔ every causal pair has L(cause) < L(effect)")
print(f"❌ but L(A: indep A1)={L['A: indep A1']} < L(C: indep C1)={L['C: indep C1']}, "
      f"and those two events are unrelated.")
print("   A single number cannot say 'these are incomparable'. Notebook 2 fixes that.")


### ✅ What Lamport gets right

Every causally-related pair obeys `L(A) < L(B)`. The write `x=1` always has a smaller Lamport time than the write `y=2`, which always has a smaller Lamport time than `z=3`. Good.

### ❌ The trap

`A: indep A1` and `C: indep C1` are **concurrent** — neither caused the other. But Lamport gives them *different* timestamps, and if you sort by Lamport time you'd conclude one came before the other.

That's the fundamental limit: **Lamport clocks give you a total order, but they can't distinguish causality from coincidence.**

In Notebook 2 we fix this with **vector clocks**.

## 3. 🎨 Visualising happens-before

A space-time diagram is the classic way to see causality. Each process is a vertical line (time flows down). Arrows show messages.

Below we draw the timeline and **label events with Lamport timestamps**. The arrows are the `→` relation.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
procs = {"A": 0, "B": 1, "C": 2}

# Events: (proc, y-position, lamport, label)
events = [
    ("A", 1, 1, "x=1"),
    ("B", 2, 2, "recv x=1"),
    ("B", 3, 3, "y=2"),
    ("C", 4, 4, "recv y=2"),
    ("C", 5, 5, "z=3"),
    ("A", 6, 2, "indep A1"),
    ("C", 6, 6, "indep C1"),
]

# Vertical process lines
for name, x in procs.items():
    ax.plot([x, x], [0, 7], color="lightgray", zorder=0)
    ax.text(x, -0.3, name, ha="center", fontsize=14, fontweight="bold")

# Dots + labels
for proc, y, L, label in events:
    x = procs[proc]
    ax.plot(x, y, "o", color="steelblue", markersize=10, zorder=3)
    ax.annotate(f"L={L}  {label}", (x, y), xytext=(10, 0),
                textcoords="offset points", va="center", fontsize=9)

# Message arrows (happens-before across processes)
ax.annotate("", xy=(1, 2), xytext=(0, 1),
            arrowprops=dict(arrowstyle="->", color="crimson", lw=1.5))
ax.annotate("", xy=(2, 4), xytext=(1, 3),
            arrowprops=dict(arrowstyle="->", color="crimson", lw=1.5))

ax.set_ylim(7.5, -0.7)  # time flows down
ax.set_xlim(-0.5, 3.0)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Space-time diagram  —  red arrows = happens-before across processes")
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout()
plt.show()


Notice that `A: indep A1` (Lamport=2) and `C: indep C1` (Lamport=6) live on **different vertical lines with no arrow connecting them**. They're concurrent. But if you sort by `L`, `A1` looks "earlier than" `C1` — a lie caused by the scalar clock collapsing two independent timelines into one axis.

This is what vector clocks are going to fix.

## 🧠 Mini-exercises

Work these out on paper, then optionally tweak the code to check.

1. In the Lamport run above, what's the Lamport timestamp of `A: write x=1`? Of `B: recv x=1`? Why isn't it just `1 + 1 = 2`?
2. If `A` and `B` never exchanged messages, could `L(A's event) < L(B's event)` still hold? What would it mean?
3. Add a line where `A` does `A.local("indep A2")` *after* `indep A1` but without messaging anyone. What Lamport timestamp does it get, and is it concurrent with `C: indep C1`? (Lamport can't tell — that's the point.)

Next up: **Notebook 2 — vector clocks**, which *can* tell.